In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
import torch
import torchvision.transforms as transforms
import torchvision.models as models
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
import random

In [ ]:
def set_seed(seed=42):  #фиксируем сиды
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [ ]:
!mkdir -p ~/.kaggle && echo KGAT_7d3d8b9d02289ce21e300dd13afac570 > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

In [ ]:
path = kagglehub.competition_download('dog-breed-identification')

print("Path to competition files:", path)
print(os.listdir(path))
labels = pd.read_csv(f"{path}/labels.csv")
print(labels.head())

In [ ]:
train_path = os.path.join(path, "train")
test_path = os.path.join(path, "test")
print(train_path)
print(test_path)

In [ ]:
#collecting full paths to images
train = []
for root, dirs, files in os.walk(train_path):
    for file in files:
        full_path = os.path.join(root, file)
        train.append(full_path)
print(len(train))
print(*train[:10], sep='\n')

In [ ]:
#это фан ячейка
i = 3
img = Image.open(train[i])
img_path = train[i]
img_id = os.path.splitext(os.path.basename(img_path))[0]
plt.imshow(img)
plt.title(" ".join(labels.loc[labels['id'] == img_id, "breed"].values[0].split("_")).title())
plt.axis("off")

In [ ]:
encoder = LabelEncoder()
labels['breed_encoded'] = encoder.fit_transform(labels['breed'])  #если дальше будем пользоваться softmax на выходе
labels.head()
labels.columns

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.12), ratio=(0.3, 3.3))
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [ ]:
class DogDataset(Dataset): #именно такие три функции из документации по датасетам
    def __init__(self, file_paths, labels, transform=None): #трансформ это из предыдущей ячейки преобразования над картинкой
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform
    def __getitem__(self, i):
        img_path = self.file_paths[i]
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        img = Image.open(img_path).convert('RGB') #перестраховочка от греха подальше
        if self.transform:
            img = self.transform(img)
        label = self.labels[img_id]
        return img, label
    def __len__(self):
        return len(self.file_paths)

In [ ]:
path_to_label = dict(zip(labels['id'], labels['breed_encoded']))
train_labels_for_split = [path_to_label[os.path.splitext(os.path.basename(img_path))[0]] for img_path in train]

In [ ]:
NUM_CLASSES = len(labels['breed'].unique())

In [ ]:
# Хотим сохранить знания модели о данных, на которых она уже обучена,
#  в исходных весах обучение шло на 1000 классов, а у нас 120
# поэтому нам не совсем подходит старый классификатор, поэтому
# мы создаем новый классификатор, который из in_features сделает num_classes

def build_model(num_classes):
    model = models.vit_b_16(weights='IMAGENET1K_V1')

    # Замораживаем все слои
    for param in model.parameters():
        param.requires_grad = False

    # Размораживаем голову, чтобы классификатор, который мы щас заменим, обучался
    for param in model.heads.parameters():
        param.requires_grad = True

    # Меняем классификатор
    in_features = model.heads.head.in_features
    # Dropout, чтобы модель не заучивала train, а учила более общие признаки
    model.heads.head = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )

    return model


In [ ]:
import copy

# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.backends.mps.is_available():
    device = torch.device('mps')
elif torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(device)

checkpoint_dir = 'checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

# Гиперпараметры
N_FOLDS = 3
EPOCHS = 15
BATCH_SIZE = 32
PATIENCE = 5
MIN_DELTA = 0.001

# чтобы моделька не сильно уверенной была в выводах
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)



In [ ]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

train = sorted(
    train)  # чтобы данные шли в одном порядке всегда, вне зависимости от файловой системы или ОС или еще чего-то
train_files_arr = np.array(train)

# собрали метки по путям картинок, т.к StratifiedKFold пытается сохранить одинаковое распределение классов в каждом фолде, ему нужно передавать и X и y, чтобы он балансировал классы 
train_lbls = [
    path_to_label[os.path.splitext(os.path.basename(img_path))[0]]
    for img_path in train]

for fold, (train_idx, val_idx) in enumerate(skf.split(train_files_arr, train_lbls)):
    print(f"\n{'=' * 60}")
    print(f"  FOLD {fold + 1} / {N_FOLDS}")
    print(f"{'=' * 60}")

    fold_train_paths = train_files_arr[train_idx].tolist()
    fold_val_paths = train_files_arr[val_idx].tolist()

    train_dataset = DogDataset(fold_train_paths, path_to_label, transform=train_transform)
    val_dataset = DogDataset(fold_val_paths, path_to_label, transform=val_transform)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=2)

    model = build_model(NUM_CLASSES).to(device)

    optimizer = optim.Adam([
        {'params': model.encoder.parameters(), 'lr': 1e-5},  # медленный чтобы модель не забыла imagenet
        {'params': model.heads.parameters(), 'lr': 1e-4},  # быстрее для обучения
    ])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=2
    )

    best_val_loss = float('inf')
    patience_counter = 0
    best_weights = copy.deepcopy(model.state_dict())

    for epoch in range(EPOCHS):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for images, labels_batch in train_loader:
            images = images.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()  #item переводит тензор в число
            predicted = outputs.argmax(dim=1)  #порода, dim1 -  ищем максимум по столбцам
            correct += (predicted == labels_batch).sum().item()
            total += labels_batch.size(0)

        train_loss = total_loss / len(train_loader)
        train_acc = correct / total * 100

        model.eval()  #переключаем модель в режим инференса
        val_loss = 0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels_batch in val_loader:
                images = images.to(device)
                labels_batch = labels_batch.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels_batch)

                val_loss += loss.item()  #item переводит тензор в число
                predicted = outputs.argmax(dim=1)  #порода, dim1 -  ищем максимум по столбцам
                val_correct += (predicted == labels_batch).sum().item()
                val_total += labels_batch.size(0)

        val_loss = val_loss / len(val_loader)
        val_acc = val_correct / val_total * 100
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        print(f"Epoch {epoch + 1:>2} | LR: {current_lr:.2e} | "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.2f}%")

        if val_loss < best_val_loss - MIN_DELTA:
            best_val_loss = val_loss
            patience_counter = 0
            best_weights = copy.deepcopy(model.state_dict())
            print('val улучшился, сохранили веса')

            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_loss,
                'val_acc': val_acc,
            }, f'{checkpoint_dir}/checkpoint_epoch{epoch + 1}_val{val_acc:.2f}.pth')
        else:
            patience_counter += 1
            print(f"нет улучшения, patience {patience_counter}/{PATIENCE}")
            if patience_counter >= PATIENCE:
                print(f"early stopping на эпохе {epoch + 1}")
                break

    model.load_state_dict(best_weights)
    print(f"Загрузили лучшие веса фолда {fold + 1}, val_loss = {best_val_loss:.4f}")

In [ ]:
class DogTestDataset(Dataset):
    def __init__(self, file_paths, transform=None):
        self.file_paths = file_paths
        self.transform = transform

    def __getitem__(self, i):
        img_path = self.file_paths[i]
        img = Image.open(img_path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        img_id = os.path.splitext(os.path.basename(img_path))[0]
        return img, img_id

    def __len__(self):
        return len(self.file_paths)


# трансформации для теста = как для валидации
test_transform = val_transform

# собираем пути к тестовым изображениям
test_files = []
for root, dirs, files in os.walk(test_path):
    for file in files:
        test_files.append(os.path.join(root, file))

test_files = sorted(test_files)

test_dataset = DogTestDataset(test_files, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
fold_probs = []
fold_ids = []

# по сути матрица предиктов разных фолдов, по которым потом будем усреднять
oof_test_probs = np.zeros((len(test_files), NUM_CLASSES))

with torch.no_grad():
    for images, img_ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)

        fold_probs.append(probs.cpu().numpy())
        fold_ids.extend(img_ids)

    fold_probs = np.vstack(fold_probs)
    oof_test_probs += fold_probs / N_FOLDS  # усредняем по фолдам

print("\nОбучение завершено!")

submission = pd.DataFrame(oof_test_probs, columns=encoder.classes_)
submission.insert(0, "id", fold_ids)

submission.to_csv("submission.csv", index=False)
print(submission.head())
print("submission.csv saved")

In [ ]:
import os

print(os.getcwd())
print(os.path.exists("submission.csv"))
from google.colab import files

files.download("/content/submission.csv")